# 05 指标计算模块 (core.metrics)

提供分类、回归、特征评估、稳定性、金融风控等场景的评估指标。

**指标分类：**
- 分类指标: ks, auc, gini, accuracy, precision, recall, f1, ks_bucket, confusion_matrix, classification_report
- 特征评估: iv, iv_table, chi2_test, cramers_v, feature_importance
- 稳定性: psi, psi_table, psi_rating, csi, csi_table, batch_psi
- 金融风控: lift, lift_at, lift_table, lift_curve, badrate, badrate_by_group
- 回归指标: mse, mae, rmse, r2

**数据说明**: 基于 `hscredit_yyp.xlsx`，目标变量为 `MOB1 > 3`

In [ ]:
import os, sys
sys.path.append('../')

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hscredit import init_setting
from hscredit.core.metrics import (
    ks, auc, gini,
    accuracy, precision, recall, f1,
    confusion_matrix, classification_report,
    ks_bucket, roc_curve,
    iv, iv_table,
    chi2_test, cramers_v,
    psi, psi_table, psi_rating,
    csi, csi_table,
    lift, lift_at, lift_table, lift_curve,
    badrate, badrate_by_group,
    mse, mae, rmse, r2,
)
from hscredit.core.models import LogisticRegression

init_setting()

df = pd.read_excel('hscredit_yyp.xlsx')
df['target'] = (df['MOB1'] > 3).astype(int)

# 预测概率：归一化中智小牛分C3
y_true = df['target'].values
y_prob = df['中智小牛分C3'].fillna(df['中智小牛分C3'].median()).values
y_prob = (y_prob - y_prob.min()) / (y_prob.max() - y_prob.min())

# 二值预测
threshold = 0.5
y_pred = (y_prob > threshold).astype(int)

print(f"样本数: {len(df):,}")
print(f"坏样本率: {y_true.mean():.2%}")

## 1. 分类指标

In [ ]:
# KS / AUC / Gini
print("=== 基础分类指标 ===")
print(f"KS:  {ks(y_true, y_prob):.4f}")
print(f"AUC: {auc(y_true, y_prob):.4f}")
print(f"Gini: {gini(y_true, y_prob):.4f}")

# ROC曲线数据
fpr, tpr, thresholds = roc_curve(y_true, y_prob)
print(f"\nROC曲线点数: {len(fpr)}")

In [ ]:
# Accuracy / Precision / Recall / F1（基于阈值预测）
print("=== 二分类评估指标 ===")
print(f"Accuracy:  {accuracy(y_true, y_pred):.4f}")
print(f"Precision: {precision(y_true, y_pred):.4f}")
print(f"Recall:    {recall(y_true, y_pred):.4f}")
print(f"F1:        {f1(y_true, y_pred):.4f}")

# 混淆矩阵
cm = confusion_matrix(y_true, y_pred)
print("\n混淆矩阵:")
print(cm)

# 分类报告
print("\n分类报告:")
print(classification_report(y_true, y_pred, target_names=['好样本', '坏样本']))

In [ ]:
# KS分箱表
ks_df = ks_bucket(y_true, y_prob, n_bins=10)
print("=== KS分箱表 ===")
display(ks_df)

## 2. 特征评估指标

In [ ]:
# IV计算（批量）
features = ['中智小牛分C3', '珊瑚92', '极光欺诈分6v1', '青云24', '占信V3']
iv_results = []
for feat_name in features:
    feat_values = df[feat_name].fillna(df[feat_name].median())
    iv_val = iv(y_true, feat_values, max_n_bins=10)
    iv_results.append({'特征': feat_name, 'IV': iv_val})

iv_results_df = pd.DataFrame(iv_results).sort_values('IV', ascending=False)
print("=== 特征IV值 ===")
display(iv_results_df)

In [ ]:
# IV详细表
iv_detail = iv_table(y_true, df['中智小牛分C3'].fillna(df['中智小牛分C3'].median()), max_n_bins=10)
print("=== IV分箱详情表 ===")
display(iv_detail)

In [ ]:
# 卡方检验和Cramers V
feat_vals = df['中智小牛分C3'].fillna(df['中智小牛分C3'].median())
chi2_result = chi2_test(feat_vals, y_true, n_bins=5)
cramers = cramers_v(feat_vals, y_true, n_bins=5)
print(f"卡方检验: chi2={chi2_result['chi2']:.4f}, p-value={chi2_result['p_value']:.4f}")
print(f"Cramers V: {cramers:.4f}")

In [ ]:
# 特征重要性（用逻辑回归模型计算）
df_clean = df[features + ['target']].dropna()
X_feat = df_clean[features]
y_feat = df_clean['target']

lr_model = LogisticRegression(max_iter=500)
lr_model.fit(X_feat, y_feat)

importance = feature_importance(lr_model, X_feat)
print("=== 特征重要性 ===")
display(importance)

## 3. 稳定性指标 (PSI/CSI)

In [ ]:
# 模拟训练集/测试集PSI
np.random.seed(42)
score_train = np.random.normal(650, 80, 500)
score_test = np.random.normal(640, 85, 300)  # 模拟分布偏移

# PSI计算
psi_val = psi(score_train, score_test, n_bins=10)
print(f"PSI: {psi_val:.4f}")

# PSI详细表
psi_detail = psi_table(score_train, score_test, n_bins=10)
print("\n=== PSI分箱详情 ===")
display(psi_detail)

In [ ]:
# PSI评级
psi_rating_result = psi_rating(psi_val)
print(f"PSI评级: {psi_rating_result}")

# CSI计算（Conditional PSI，条件PSI）
csi_val = csi(score_train, score_test, y_train=np.ones(500), y_test=np.ones(300), n_bins=10)
print(f"CSI: {csi_val:.4f}")

# CSI分箱表
csi_detail = csi_table(score_train, score_test, y_train=np.ones(500), y_test=np.ones(300), n_bins=10)
print("\n=== CSI分箱详情 ===")
display(csi_detail)

## 4. 金融风控指标

In [ ]:
# Lift指标
print("=== Lift指标 ===")
for ratio in [0.01, 0.03, 0.05, 0.10]:
    lift_val = lift_at(y_true, y_prob, ratios=ratio)
    print(f"Lift@{ratio*100:.0f}%: {lift_val:.2f}")

# lift_table
lift_t = lift_table(y_true, y_prob, n_bins=10)
print("\n=== Lift分箱表 ===")
display(lift_t)

In [ ]:
# Lift曲线
lift_curve_df = lift_curve(y_true, y_prob)
plt.figure(figsize=(10, 4))
plt.plot(lift_curve_df['覆盖率'] * 100, lift_curve_df['Lift值'], 'o-', linewidth=2)
plt.axhline(y=1, color='red', linestyle='--')
plt.xlabel('覆盖率 (%)')
plt.ylabel('Lift值')
plt.title('Lift曲线')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 坏样本率指标
feat_for_br = df['中智小牛分C3'].fillna(df['中智小牛分C3'].median())

# 按特征分箱的坏样本率
br_by_bin = badrate(feat_for_br, y_true, n_bins=5)
print("=== 特征分箱坏样本率 ===")
display(br_by_bin)

# 按自定义分组的坏样本率
df['score_group'] = pd.qcut(feat_for_br, q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
br_by_group = badrate_by_group(df['score_group'], y_true)
print("\n=== 分组坏样本率 ===")
display(br_by_group)

## 5. 回归指标

In [ ]:
# 模拟回归场景
y_true_reg = np.array([3.2, 4.5, 2.8, 5.1, 3.9, 4.2, 2.5, 5.5, 3.0, 4.8])
y_pred_reg = np.array([3.0, 4.3, 3.0, 5.0, 4.1, 4.0, 2.7, 5.3, 3.2, 4.6])

print("=== 回归指标 ===")
print(f"MSE:  {mse(y_true_reg, y_pred_reg):.4f}")
print(f"MAE:  {mae(y_true_reg, y_pred_reg):.4f}")
print(f"RMSE: {rmse(y_true_reg, y_pred_reg):.4f}")
print(f"R2:   {r2(y_true_reg, y_pred_reg):.4f}")

## 6. 指标汇总对比

In [ ]:
# 批量IV对比
print("=== 特征IV值排序 ===")
display(iv_results_df)

plt.figure(figsize=(10, 5))
plt.barh(iv_results_df['特征'], iv_results_df['IV'])
plt.xlabel('IV值')
plt.title('特征IV值对比')
plt.tight_layout()
plt.show()